# Ethiopia Climate Data: Profiling, Cleaning, and EDA

Analyze Ethiopia climate data to identify temperature, precipitation, and extreme-weather patterns relevant to climate resilience planning in the lead-up to COP32.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


## 1) Data Loading and Date Parsing
The dataset provides `YEAR` and `DOY` (day-of-year). We build a proper `DATE` column for time-series analysis.

In [ ]:
df = pd.read_csv('ethiopia.csv', header=0)
df['Country'] = 'Ethiopia'

# Replace NASA sentinel -999 values with NaN before any statistics or profiling
df.replace(-999, np.nan, inplace=True)

# Convert YEAR and DOY into a proper datetime column, then extract month
df["DATE"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df["MONTH"] = df["DATE"].dt.month

print('Shape:', df.shape)
print('Date range:', df['DATE'].min().date(), 'to', df['DATE'].max().date())
df.head()


### NASA Header and Sentinel Handling
This dataset is loaded with `pd.read_csv(..., header=0)` so the first row is treated as the column header. Any NASA-style metadata header rows would need explicit `skiprows` handling, but the file here is already aligned to the standard tabular format.

Sentinel values of `-999` are converted to `NaN` immediately so summary statistics and missing-value logic treat them as missing data rather than valid observations.

The cleaned file is exported later to `data/ethiopia_clean.csv`, and `data/` is excluded from version control via `.gitignore`.

## 2) Data Profiling
Profile the dataset and compute missing-value percentages for each column.

In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
})
missing_summary[missing_summary["missing_count"] > 0]


### Missing-Value Report
List any column with greater than 5% nulls and note what that implies for analysis.

## 3) Duplicate Detection
Identify and remove exact duplicate rows before further cleaning.

In [ ]:
duplicate_mask = df.duplicated(keep=False)
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)
if duplicates:
    print("Columns involved in exact duplicate rows:", list(df.columns))
    display(df[duplicate_mask].head(5))
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)


### Duplicate Handling Rationale
Removing exact duplicate rows ensures the cleaned dataset does not over-count repeated observations. This protects statistical summaries, trend estimates, and outlier detection from duplicated records that would otherwise distort results.

### Duplicate Detection Result
Exact duplicates are removed from the raw dataset to avoid bias in downstream analyses.


## 4) Outlier Detection & Basic Cleaning
Compute Z-scores for key climate variables, handle outliers, and clean remaining missing values.

In [ ]:
clean_df = df.copy()
weather_cols = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]
for col in weather_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

clean_df = clean_df.drop_duplicates().sort_values("DATE").reset_index(drop=True)

z_scores = clean_df[weather_cols].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
outlier_mask = z_scores.abs() > 3
outlier_count = outlier_mask.any(axis=1).sum()
print("Rows flagged as outliers (|Z|>3):", outlier_count)
print("Outlier counts by variable:")
print(outlier_mask.sum())

clean_df["outlier_flag"] = outlier_mask.any(axis=1)

# Retain flagged outliers because extreme climate values can be valid signals,
# especially in climate resilience and adaptation analysis.
clean_df[weather_cols] = clean_df[weather_cols].ffill()
row_missing_pct = clean_df.isna().mean(axis=1)
rows_to_drop = (row_missing_pct > 0.3).sum()
print("Rows with >30% missing values to drop:", rows_to_drop)
clean_df = clean_df[row_missing_pct <= 0.3].reset_index(drop=True)

os.makedirs("data", exist_ok=True)
clean_df.to_csv("data/ethiopia_clean.csv", index=False)
print("Exported cleaned DataFrame to data/ethiopia_clean.csv")


### Outlier and Missing-Value Cleaning Decisions
- Outliers are computed for the requested variables: `T2M`, `T2M_MAX`, `T2M_MIN`, `PRECTOTCORR`, `RH2M`, `WS2M`, and `WS2M_MAX`.
- We flag rows where `|Z| > 3` but retain them rather than dropping or capping, because extreme values can represent real climate events important for impact and adaptation analysis.
- Remaining missing weather values are filled forward to preserve continuity in the time series and avoid introducing artificial discontinuities.
- Rows with more than 30% missing values are dropped because they are too incomplete to impute reliably and would weaken downstream analysis.
- The cleaned dataset is exported to `data/ethiopia_clean.csv` and `data/` is already ignored in `.gitignore`.


## 5) Feature Engineering for Trend Analysis
Create monthly and yearly aggregates for focused climate trend analysis.

In [ ]:
clean_df['YEAR_INT'] = clean_df['DATE'].dt.year
clean_df['MONTH'] = clean_df['DATE'].dt.month

yearly = clean_df.groupby('YEAR_INT', as_index=False).agg(
    avg_temp=('T2M', 'mean'),
    max_temp=('T2M_MAX', 'mean'),
    min_temp=('T2M_MIN', 'mean'),
    total_precip=('PRECTOTCORR', 'sum'),
    avg_wind=('WS2M', 'mean'),
    max_wind=('WS2M_MAX', 'max')
)

yearly.head()

### Time Series Interpretation
The annual average temperature trend highlights multi-year warming patterns. A rising series suggests increasing heat stress across Ethiopia, which is critical for climate adaptation planning in agriculture and water resources.

### Time Series Interpretation
The annual precipitation trend shows whether wet-season totals are increasing or declining. Large year-to-year swings indicate hydrological variability that can challenge water management and agricultural planning.

### Temperature Trend Interpretation
The separate max/min temperature series help reveal whether warming is concentrated in daytime highs, nighttime lows, or both. A widening gap may indicate increasing diurnal temperature range, while a parallel rise suggests broad warming.

## 6) Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(yearly['YEAR_INT'], yearly['avg_temp'], marker='o')
plt.title('Ethiopia: Yearly Average Temperature (T2M)')
plt.xlabel('Year')
plt.ylabel('Temperature (C)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(yearly['YEAR_INT'], yearly['total_precip'], marker='o', color='teal')
plt.title('Ethiopia: Yearly Total Precipitation (PRECTOTCORR)')
plt.xlabel('Year')
plt.ylabel('Total Precipitation')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(yearly['YEAR_INT'], yearly['max_temp'], color='tomato')
ax[0].set_title('Yearly Mean of Daily Max Temp (T2M_MAX)')
ax[0].set_xlabel('Year')
ax[0].set_ylabel('Temperature (C)')

ax[1].plot(yearly['YEAR_INT'], yearly['min_temp'], color='royalblue')
ax[1].set_title('Yearly Mean of Daily Min Temp (T2M_MIN)')
ax[1].set_xlabel('Year')
ax[1].set_ylabel('Temperature (C)')

plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']

plt.figure(figsize=(9, 6))
sns.heatmap(clean_df[corr_cols].corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap of Key Climate Variables')
plt.tight_layout()
plt.show()

### Correlation Interpretation
The heatmap identifies how temperature, precipitation, humidity, wind, pressure, and moisture variables move together. Strong relationships point to climate system linkages, while weak correlations suggest independent drivers that may require separate adaptation strategies.

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(clean_df['T2M'].dropna(), bins=30, kde=True, color='navy')
plt.title('Distribution of Daily Mean Temperature (T2M)')
plt.xlabel('Temperature (C)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

### Distribution Interpretation
The temperature distribution plot reveals the central tendency and the frequency of extreme warm days. A pronounced right tail would highlight how often unusually high temperatures occur, supporting heat-risk and adaptation planning.

## 7) Change Comparison: Early vs Recent Period
Compare averages between early years and recent years to highlight directional climate shifts.

In [ ]:
early = clean_df[clean_df['YEAR_INT'].between(2015, 2019)]
recent = clean_df[clean_df['YEAR_INT'].between(2022, 2026)]

comparison = pd.DataFrame({
    'metric': ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'WS2M_MAX'],
    'early_mean': [early['T2M'].mean(), early['T2M_MAX'].mean(), early['T2M_MIN'].mean(), early['PRECTOTCORR'].mean(), early['WS2M_MAX'].mean()],
    'recent_mean': [recent['T2M'].mean(), recent['T2M_MAX'].mean(), recent['T2M_MIN'].mean(), recent['PRECTOTCORR'].mean(), recent['WS2M_MAX'].mean()]
})
comparison['delta_recent_minus_early'] = comparison['recent_mean'] - comparison['early_mean']
comparison

## 8) COP32-Focused Insights
1. **Warming signal**: Annual average temperature and extreme temperature indicators suggest persistent heat pressure.
2. **Precipitation variability**: Year-to-year precipitation totals fluctuate, indicating potential water-planning uncertainty.
3. **Extreme conditions**: Maximum wind and temperature indicators support adaptation planning for infrastructure and agriculture.
4. **Policy relevance**: Trends reinforce the need for resilient agriculture, water management, and heat-risk preparedness in Ethiopia.

## 9) References and Self-Learning Notes
- pandas `read_csv`, missing-value handling, and DataFrame cleaning: https://pandas.pydata.org/docs/
- Z-score outlier detection and standard score interpretation: https://en.wikipedia.org/wiki/Standard_score
- Seaborn visualization reference: https://seaborn.pydata.org/
- Climate trend interpretation principles: use trends, variability, and extremes to inform adaptation policy.

This notebook was informed by official pandas and seaborn documentation, plus general time-series and climate-data best practices.

## 10) Next Analytical Steps
- Repeat the exact pipeline for Kenya, Nigeria, Sudan, and Tanzania.
- Add anomaly analysis (against baseline years) for stronger trend interpretation.
- Combine country-level outputs into a cross-country comparative COP32 briefing.